In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


#  MILESTONE 2

# Introduction to Hugging Face transformers and datasets

In [2]:
import torch, numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, pipeline
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATA = "/kaggle/input/competitions/smart-mcq-solver-challenge"
OPTIONS = list("ABCDE")
DEV = 0 if torch.cuda.is_available() else -1

**Q1: Load train.csv using the Hugging Face datasets library (do not use pandas). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. E.g., prompt_text A_text. What is the exact character length (total number of string characters using Python's len() function, NOT the number of tokens) of the combined_text string for the row at index 51? Note: We follow zero-indexing here.**


In [3]:
raw = load_dataset("csv", data_files=f"{DATA}/train.csv")["train"]
raw = raw.map(lambda x: {"combined_text": x["prompt"] + " " + x["A"]})
Q1 = len(raw[51]["combined_text"])
print("Q1:", Q1)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Q1: 614


**Q2: Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer?**

**Q3: Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token.**


In [4]:
prompts = [str(x) for x in raw["prompt"]]
opts    = {o: [str(x) for x in raw[o]] for o in OPTIONS}
answers = [str(x) for x in raw["answer"]]

In [5]:
bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
Q2 = bert_tok.vocab_size
Q3 = bert_tok.sep_token_id
print("Q2:", Q2)
print("Q3:", Q3)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q2: 30522
Q3: 102


**Q4: Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors).** 

**What is the exact geometric shape (dimensions) of the resulting input_ids tensor?**

In [6]:
enc = bert_tok(prompts, padding="max_length", truncation=True,
               max_length=128, return_tensors="pt")
Q4 = tuple(enc["input_ids"].shape)
print("Q4:", Q4)

Q4: (2000, 128)


# BERT/RoBERTa Architecture & Attention Mechanisms

**Q5: Load train.csv using the Hugging Face datasets library (do not use pandas). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. E.g., prompt_text A_text. What is the exact character length (total number of string characters using Python's len() function, NOT the number of tokens) of the combined_text string for the row at index 51? Note: We follow zero-indexing here.**

In [7]:
Q5 = 768 // 12
print("Q5:", Q5)

Q5: 64


**Q6: Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer?**

**Q7: Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token.**

In [8]:
bert = AutoModel.from_pretrained("bert-base-uncased").eval()
inp0 = bert_tok(prompts[0], return_tensors="pt")
with torch.no_grad():
    out0 = bert(**inp0)
Q6 = tuple(out0.last_hidden_state.shape)
cls_vec = out0.last_hidden_state[0, 0]
Q7 = round(cls_vec[:5].sum().item(), 4)
print("Q6:", Q6)
print("Q7:", Q7)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Q6: (1, 31, 768)
Q7: -1.2001


**Q8: Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors).**

**What is the exact geometric shape (dimensions) of the resulting input_ids tensor?**


In [9]:
bert_att = AutoModel.from_pretrained("bert-base-uncased",
                                     output_attentions=True).eval()
inp = bert_tok("Light-ion fusion is a technique.", return_tensors="pt")
with torch.no_grad():
    out = bert_att(**inp)
att = out.attentions[-1][0, 0]                      
toks = bert_tok.convert_ids_to_tokens(inp["input_ids"][0])
fusion_idx = toks.index("fusion")
Q8 = round(att[0, fusion_idx].item(), 4)    
print("Q8:", Q8)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Q8: 0.1025


# BERT/RoBERTa Architecture & Attention Mechanisms


**Q9: A standard bert-base-uncased model has a hidden embedding size of 768 dimensions and uses exactly 12 attention heads in each layer.**

**In Transformer architecture, the hidden size is divided equally among the attention heads. What is the exact dimensionality (size) of each individual attention head?**


In [10]:
minilm = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
e_p = minilm.encode(prompts[0])
e_b = minilm.encode(opts["B"][0])
Q9 = round(util.cos_sim(e_p, e_b).item(), 4)
print("Q9:", Q9)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Q9: 0.7658


**Q10: Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt from row ID 0 using the tokenizer's default settings (do not apply any manual padding or truncation). Pass this tokenized input through the model. Look at the output object.** 

**What is the exact shape of the last_hidden_state tensor returned?**

*Note: We follow zero-indexing here.*


In [11]:
def top3_letters(scores):
    return [OPTIONS[j] for j in np.argsort(scores)[::-1][:3]]
def map3(top3_list, truths):
    s = 0.0
    for preds, t in zip(top3_list, truths):
        for i, p in enumerate(preds):
            if p == t:
                s += 1.0 / (i + 1); break
    return s / len(truths)

# MiniLM pipeline
p_emb = minilm.encode(prompts, batch_size=64, normalize_embeddings=True,
                      convert_to_numpy=True, show_progress_bar=True)
o_emb = {o: minilm.encode(opts[o], batch_size=64, normalize_embeddings=True,
                          convert_to_numpy=True) for o in OPTIONS}
minilm_top3 = [top3_letters([float(p_emb[i] @ o_emb[o][i]) for o in OPTIONS])
               for i in range(len(answers))]
minilm_map3 = map3(minilm_top3, answers)

# TF-IDF pipeline 
corpus = prompts + sum([opts[o] for o in OPTIONS], [])
vec = TfidfVectorizer(stop_words="english").fit(corpus)
tfidf_top3 = []
for i in range(len(answers)):
    pv = vec.transform([prompts[i]])
    ov = vec.transform([opts[o][i] for o in OPTIONS])
    tfidf_top3.append(top3_letters(cosine_similarity(pv, ov).ravel()))

count = sum(1 for i, t in enumerate(answers)
            if t not in tfidf_top3[i] and t in minilm_top3[i])
Q10_map3, Q10_count = round(minilm_map3, 4), count
print("Q10_count:", Q10_count)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Q10_count: 575


# Zero-shot classification concepts 


**Q11: Using the last_hidden_state tensor from the previous question, extract the embedding vector representing the [CLS] token (which is always the token at index 0). What is the sum of the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places).**


**Q12: Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0).** 

**What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places).**


In [12]:
zs = pipeline("zero-shot-classification", device=DEV)
cand = [opts["A"][1], opts["B"][1], opts["C"][1]]
r1 = zs(prompts[1], candidate_labels=cand)                    # softmax
r2 = zs(prompts[1], candidate_labels=cand, multi_label=True)  # sigmoid
Q11 = round(r1["scores"][0], 4)
Q12 = round(abs(sum(r1["scores"]) - sum(r2["scores"])), 4)
print("Q11:", Q11)
print("Q12:", Q12)

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q11: 0.4575
Q12: 0.9995


# Context-Aware Embeddings

**Q13: Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Use the model's .encode() method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the cosine similarity between these two vectors specifically using the sentence_transformers.util.cos_sim() function. What is the resulting similarity score rounded to 4 decimal places? Note: We follow zero-indexing here.**


In [13]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

t5_tok = AutoTokenizer.from_pretrained("google/flan-t5-small")
t5 = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")
if DEV == 0:
    t5 = t5.to("cuda")

s = (f'Question: {prompts[0]}. Is the correct answer A: {opts["A"][0]} '
     f'or B: {opts["B"][0]}? Answer with just the letter A or B.')

inp = t5_tok(s, return_tensors="pt")
if DEV == 0:
    inp = {k: v.to("cuda") for k, v in inp.items()}

out = t5.generate(**inp, max_new_tokens=5)
Q13 = t5_tok.decode(out[0], skip_special_tokens=True)
print("Q13:", repr(Q13))

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Q13: 'B'


In [14]:
print("MILESTONE 2 - ANSWERS")
for k, v in dict(Q1=Q1, Q2=Q2, Q3=Q3, Q4=Q4, Q5=Q5, Q6=Q6, Q7=Q7,
                 Q8=Q8, Q9=Q9, Q10_MiniLM_MAP3=Q10_map3, Q10_count=Q10_count,
                 Q11=Q11, Q12=Q12, Q13=repr(Q13)).items():
    print(f"{k}: {v}")

MILESTONE 2 - ANSWERS
Q1: 614
Q2: 30522
Q3: 102
Q4: (2000, 128)
Q5: 64
Q6: (1, 31, 768)
Q7: -1.2001
Q8: 0.1025
Q9: 0.7658
Q10_MiniLM_MAP3: 0.4231
Q10_count: 575
Q11: 0.4575
Q12: 0.9995
Q13: 'B'
